In [1]:
import pandas as pd
import onca_utils as ou
import numpy as np
import os

#import onca_products as op

In [2]:
# I/O paths
input_conapo_poblaciones = "/data3/onca_lite/requirements/poblaciones_group_quinq.csv"
poblaciones_who = "/data3/onca_lite/requirements/poblaciones_WHO.csv"
input_cat_entidades = "/data3/onca_lite/requirements/entidades_fed.csv"
input_cat_municipios = "/data3/onca_lite/requirements/municipios_geo.csv"
input_cat_edades = "/data3/onca_lite/requirements/EDADES.csv"
input_mortality_folder = "/data3/onca_lite/DATOS_CRUDOS/"
cie10 = "C910"

In [3]:
print("Cargando catalogos")
catalog_loader = ou.CatalogLoader()
# Archivo de poblaciones
conapo_populations = catalog_loader.load_conapo_populations(input_conapo_poblaciones)

# Catalogo de entidades y municipios
cat_entidades = catalog_loader.load_states(input_cat_entidades)
cat_municipios = catalog_loader.load_municipalities(input_cat_municipios)
# Catalogo de edades
cat_edades = catalog_loader.load_ages(input_cat_edades)
del(catalog_loader)

Cargando catalogos


/home/lpaniagua/onca_lite/src/onca_utils.py:54: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  pe = pe.replace({'Hombres':1,'Mujeres':2})


In [4]:
print("Cargando registros de mortalidad")
deaths = ou.DeathRegistryLoader().load_deaths(input_mortality_folder, cat_edades, cie10)
deaths = deaths[(deaths.ANIO_REGIS >= 2000) & (deaths.ANIO_REGIS != 9999)]

Cargando registros de mortalidad
Realizando extracción de registros para C910
Procesando archivo 2000.csv
Procesando archivo 2001.csv
Procesando archivo 2002.csv
Procesando archivo 2003.csv
Procesando archivo 2004.csv
Procesando archivo 2005.csv


/home/lpaniagua/onca_lite/src/onca_utils.py:92: DtypeWarning: Columns (49) have mixed types. Specify dtype option on import or set low_memory=False.
  deaths = pd.read_csv(input_path)


Procesando archivo 2006.csv
Procesando archivo 2007.csv
Procesando archivo 2008.csv
Procesando archivo 2009.csv
Procesando archivo 2010.csv
Procesando archivo 2011.csv
Procesando archivo 2012.csv
Procesando archivo 2013.csv


/home/lpaniagua/onca_lite/src/onca_utils.py:92: DtypeWarning: Columns (42,50) have mixed types. Specify dtype option on import or set low_memory=False.
  deaths = pd.read_csv(input_path)


Procesando archivo 2014.csv
Procesando archivo 2015.csv
Procesando archivo 2016.csv
Procesando archivo 2017.csv
Procesando archivo 2018.csv
Procesando archivo 2019.csv
Procesando archivo 2020.csv
Procesando archivo 2021.csv
Procesando archivo 2022.csv


/home/lpaniagua/onca_lite/src/onca_utils.py:92: DtypeWarning: Columns (72) have mixed types. Specify dtype option on import or set low_memory=False.
  deaths = pd.read_csv(input_path)


Procesando archivo 2023.csv


In [5]:
deaths

,ANIO_REGIS,ENT_CVE,MUN_CVE,CAUSA_DEF,SEXO,EDAD_NUM,RANGO_EDAD
0,2000,1,1,C910,2,4.0,00_04
1,2000,1,1,C910,2,15.0,15_19
2,2000,1,1,C910,1,9.0,05_09
3,2000,1,1,C910,2,38.0,35_39
4,2000,1,1,C910,2,12.0,10_14
...,...,...,...,...,...,...,...
40840,2023,32,56,C910,1,13.0,10_14
40841,2023,32,10,C910,1,41.0,40_44
40842,2023,32,56,C910,2,23.0,20_24
40843,2023,32,10,C910,2,42.0,40_44


In [6]:

mc = ou.MortalityCalculator()

In [7]:
# Config de escalas x nivel geográfico
scale_config = {
    'Municipal': {'code': 1, 'scale': '1K', 'factor': 1000},
    'Estatal': {'code': 2, 'scale': '10K', 'factor': 10000}, 
    'Nacional': {'code': 3, 'scale': '100K', 'factor': 100000}
}

# Obtener configuración de escala
def get_scale_config(nivel):
    return scale_config[nivel]
    

In [8]:
# Variación de rangos de edad
age_groups = np.array(['00_04', '05_09', '10_14', '15_19', '20_24', '25_29', '30_34',
                       '35_39', '40_44', '45_49', '50_54', '55_59', '60_64', '65_69',
                       '70_74', '75_79', '80_84', '>85'])
arr_l = age_groups.shape[0]

# Función para crear el mapeo de edad_inicio y edad_final
def create_edad_ranges():
    edad_ranges = pd.DataFrame({
        'RANGO_EDAD': ['00_04', '05_09', '10_14', '15_19', '20_24', '25_29', '30_34',
                       '35_39', '40_44', '45_49', '50_54', '55_59', '60_64', '65_69',
                       '70_74', '75_79', '80_84', '>85'],
        'EDAD_INICIO': [0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80, 85],
        'EDAD_FINAL': [4, 9, 14, 19, 24, 29, 34, 39, 44, 49, 54, 59, 64, 69, 74, 79, 84, 125]
    })
    return edad_ranges

edad_ranges = create_edad_ranges()
edad_ranges.head()

,RANGO_EDAD,EDAD_INICIO,EDAD_FINAL
0,00_04,0,4
1,05_09,5,9
2,10_14,10,14
3,15_19,15,19
4,20_24,20,24


In [9]:
sex_id = 3  # 1=Hombres, 2=Mujeres, 3=Ambos sexos


In [10]:
# Inicializar listas para almacenar resultados
all_tasas_crudas_municipal = []
all_tasas_crudas_estatal = []
all_tasas_crudas_nacional = []
all_tasas_asmr_municipal = []
all_tasas_asmr_estatal = []
all_tasas_asmr_nacional = []

for l in np.arange(arr_l) + 1:
    for i in np.arange(arr_l-l+1):
        current_age_groups = age_groups[i:i+l]
        current_edad_ranges = edad_ranges[edad_ranges.RANGO_EDAD.isin(current_age_groups)].copy()
        
        print(f"Procesando combinación {i+1}/{arr_l-l+1} para longitud {l}: grupos {current_age_groups}")
        print(f"  Rango de edades: {current_edad_ranges['EDAD_INICIO'].min()}-{current_edad_ranges['EDAD_FINAL'].max()}")
        
        # Filtrar muertes para los grupos de edad actuales
        filtered_deaths = deaths[deaths.RANGO_EDAD.isin(current_age_groups)].copy()
        
        if sex_id == 3:
            deaths_filtered = filtered_deaths.copy()
        else:
            deaths_filtered = filtered_deaths[filtered_deaths.SEXO == sex_id].copy()
        
        if len(deaths_filtered) == 0:
            print(f"    No hay datos para estos grupos, saltando...")
            continue
        
        print(f"    Registros de muertes encontrados: {len(deaths_filtered)}")
        
        # ================================================================
        # TASAS CRUDAS - NIVEL MUNICIPAL
        # ================================================================
        print(f"    Calculando tasas crudas específicas - Nivel Municipal")
        municipal_config = get_scale_config('Municipal')
        df_municipal_crudo = mc.compute_raw_mortality_rate(
            deaths_filtered,
            conapo_populations,
            ['ANIO_REGIS', 'ENT_CVE','MUN_CVE', 'SEXO', 'RANGO_EDAD'] if sex_id == 3 else ['ANIO_REGIS', 'ENT_CVE','MUN_CVE','RANGO_EDAD']
        )

        if len(df_municipal_crudo) == 0:
            print(f"      No hay datos municipales para estos grupos")
        else:
            df_municipal_crudo = df_municipal_crudo.merge(current_edad_ranges, on='RANGO_EDAD', how='left')
            rate_column = f"TASA_CRUDA_{municipal_config['scale']}"
            
            tasas_crudas_municipal = pd.DataFrame({
                'CIE10': cie10,
                'ANIO': df_municipal_crudo['ANIO_REGIS'],
                'ENFOQUE': 1,  # Municipal 
                'CVE_ENT': df_municipal_crudo['ENT_CVE'],
                'CVE_MUN': df_municipal_crudo['MUN_CVE'],
                'SEXO': df_municipal_crudo['SEXO'] if 'SEXO' in df_municipal_crudo.columns else sex_id,
                'EDAD_INICIO': current_edad_ranges['EDAD_INICIO'].min(),  
                'EDAD_FINAL': current_edad_ranges['EDAD_FINAL'].max(),     
                'CONTEO': df_municipal_crudo['DEFUNCIONES'],
                'TASA_CRUDA': df_municipal_crudo[rate_column],  
                'ASMR': 0,  
                'ESCALA': municipal_config['code']
            })
            
            all_tasas_crudas_municipal.append(tasas_crudas_municipal)
            print(f"      Datos municipales procesados: {len(tasas_crudas_municipal)} registros")
        
        # ================================================================
        # TASAS CRUDAS - NIVEL ESTATAL
        # ================================================================
        print(f"    Calculando tasas crudas específicas - Nivel Estatal")
        estatal_config = get_scale_config('Estatal')
        df_estatal_crudo = mc.compute_raw_mortality_rate(
            deaths_filtered,  
            conapo_populations,
            ['ANIO_REGIS', 'ENT_CVE', 'SEXO', 'RANGO_EDAD'] if sex_id == 3 else ['ANIO_REGIS', 'ENT_CVE', 'RANGO_EDAD']
        )

        if len(df_estatal_crudo) == 0:
            print(f"      No hay datos estatales para estos grupos")
        else:
            df_estatal_crudo = df_estatal_crudo.merge(current_edad_ranges, on='RANGO_EDAD', how='left')
            rate_column = f"TASA_CRUDA_{estatal_config['scale']}"

            tasas_crudas_estatal = pd.DataFrame({
                'CIE10': cie10,
                'ANIO': df_estatal_crudo['ANIO_REGIS'],
                'ENFOQUE': 2,  # Estatal 
                'CVE_ENT': df_estatal_crudo['ENT_CVE'],
                'CVE_MUN': None,
                'SEXO': df_estatal_crudo['SEXO'] if 'SEXO' in df_estatal_crudo.columns else sex_id,
                'EDAD_INICIO': current_edad_ranges['EDAD_INICIO'].min(),  
                'EDAD_FINAL': current_edad_ranges['EDAD_FINAL'].max(),    
                'CONTEO': df_estatal_crudo['DEFUNCIONES'],
                'TASA_CRUDA': df_estatal_crudo[rate_column],
                'ASMR': 0,  
                'ESCALA': estatal_config['code']
            })

            all_tasas_crudas_estatal.append(tasas_crudas_estatal)
            print(f"      Datos estatales procesados: {len(tasas_crudas_estatal)} registros")
        
        # ================================================================
        # TASAS CRUDAS - NIVEL NACIONAL
        # ================================================================
        print(f"    Calculando tasas crudas específicas - Nivel Nacional")
        nacional_config = get_scale_config('Nacional')
        df_nacional_crudo = mc.compute_raw_mortality_rate(
            deaths_filtered,  
            conapo_populations,
            ['ANIO_REGIS', 'SEXO', 'RANGO_EDAD'] if sex_id == 3 else ['ANIO_REGIS', 'RANGO_EDAD']
        )

        if len(df_nacional_crudo) == 0:
            print(f"      No hay datos nacionales para estos grupos")
        else:
            df_nacional_crudo = df_nacional_crudo.merge(current_edad_ranges, on='RANGO_EDAD', how='left')
            rate_column = f"TASA_CRUDA_{nacional_config['scale']}"

            tasas_crudas_nacional = pd.DataFrame({
                'CIE10': cie10,
                'ANIO': df_nacional_crudo['ANIO_REGIS'],
                'ENFOQUE': 3,  # Nacional
                'CVE_ENT': None,
                'CVE_MUN': None,
                'SEXO': df_nacional_crudo['SEXO'] if 'SEXO' in df_nacional_crudo.columns else sex_id,
                'EDAD_INICIO': current_edad_ranges['EDAD_INICIO'].min(),  
                'EDAD_FINAL': current_edad_ranges['EDAD_FINAL'].max(),    
                'CONTEO': df_nacional_crudo['DEFUNCIONES'],
                'TASA_CRUDA': df_nacional_crudo[rate_column],
                'ASMR': 0,  
                'ESCALA': nacional_config['code']
            })

            all_tasas_crudas_nacional.append(tasas_crudas_nacional)
            print(f"      Datos nacionales procesados: {len(tasas_crudas_nacional)} registros")
        
        # ================================================================
        # TASAS ESTANDARIZADAS ASMR - NIVEL MUNICIPAL
        # ================================================================
        if len(df_municipal_crudo) > 0:
            print(f"    Calculando tasas estandarizadas - Nivel Municipal")
            std_name = 'WHO'        
            scale = '1K'          
            asmr_column = f"ASR({std_name})_{municipal_config['scale']}"

            who_municipal = ou.MortalityStandardizer(
                file_path=poblaciones_who,
                std_name=std_name,
                age_groups=current_age_groups.tolist()
            )

            asmr_municipal_list = []
            group_columns = ['ANIO_REGIS', 'ENT_CVE', 'MUN_CVE', 'SEXO'] if sex_id == 3 else ['ANIO_REGIS', 'ENT_CVE', 'MUN_CVE']
            
            for group_key, group in df_municipal_crudo.groupby(group_columns):
                if len(group) > 0:
                    df_asmr_temp = who_municipal.compute_ASR(
                        df=group[['ANIO_REGIS', 'RANGO_EDAD', 'TASA_CRUDA_100K']],
                        age_column="RANGO_EDAD",
                        rate_column="TASA_CRUDA_100K",
                        scale=scale
                    )
                    
                    if sex_id == 3:
                        anio, ent, mun, sexo = group_key
                        df_asmr_temp['ANIO_REGIS'] = anio
                        df_asmr_temp['ENT_CVE'] = ent
                        df_asmr_temp['MUN_CVE'] = mun
                        df_asmr_temp['SEXO'] = sexo
                    else:
                        anio, ent, mun = group_key
                        df_asmr_temp['ANIO_REGIS'] = anio
                        df_asmr_temp['ENT_CVE'] = ent
                        df_asmr_temp['MUN_CVE'] = mun
                        df_asmr_temp['SEXO'] = sex_id
                        
                    df_asmr_temp['CONTEO_TOTAL'] = group['DEFUNCIONES'].sum()
                    asmr_municipal_list.append(df_asmr_temp)

            if asmr_municipal_list:
                df_asmr_municipal = pd.concat(asmr_municipal_list, ignore_index=True)
                
                tasas_asmr_municipal = pd.DataFrame({
                    'CIE10': cie10,
                    'ANIO': df_asmr_municipal['ANIO_REGIS'],
                    'ENFOQUE': 1,
                    'CVE_ENT': df_asmr_municipal['ENT_CVE'],
                    'CVE_MUN': df_asmr_municipal['MUN_CVE'],
                    'SEXO': df_asmr_municipal['SEXO'],
                    'EDAD_INICIO': current_edad_ranges['EDAD_INICIO'].min(),  
                    'EDAD_FINAL': current_edad_ranges['EDAD_FINAL'].max(),    
                    'CONTEO': df_asmr_municipal['CONTEO_TOTAL'],
                    'TASA_CRUDA': 0,  
                    'ASMR': df_asmr_municipal[asmr_column],
                    'ESCALA': municipal_config['code']
                })
                
                all_tasas_asmr_municipal.append(tasas_asmr_municipal)
                print(f"      ASMR municipal procesado: {len(tasas_asmr_municipal)} registros")
        
        # ================================================================
        # TASAS ESTANDARIZADAS ASMR - NIVEL ESTATAL
        # ================================================================
        if len(df_estatal_crudo) > 0:
            print(f"    Calculando tasas estandarizadas - Nivel Estatal")
            std_name = 'WHO'        
            scale = '10K'
            asmr_column = f"ASR({std_name})_{estatal_config['scale']}"

            who_estatal = ou.MortalityStandardizer(
                file_path=poblaciones_who,
                std_name=std_name,
                age_groups=current_age_groups.tolist()
            )

            asmr_estatal_list = []
            group_columns = ['ANIO_REGIS', 'ENT_CVE', 'SEXO'] if sex_id == 3 else ['ANIO_REGIS', 'ENT_CVE']
                    
            for group_key, group in df_estatal_crudo.groupby(group_columns):
                if len(group) > 0:
                    df_asmr_temp = who_estatal.compute_ASR(
                        df=group[['ANIO_REGIS', 'RANGO_EDAD', 'TASA_CRUDA_100K']],
                        age_column="RANGO_EDAD",
                        rate_column="TASA_CRUDA_100K",
                        scale=scale
                    )
                            
                    if sex_id == 3:
                        anio, ent, sexo = group_key
                        df_asmr_temp['ANIO_REGIS'] = anio
                        df_asmr_temp['ENT_CVE'] = ent
                        df_asmr_temp['SEXO'] = sexo
                    else:
                        anio, ent = group_key
                        df_asmr_temp['ANIO_REGIS'] = anio
                        df_asmr_temp['ENT_CVE'] = ent
                        df_asmr_temp['SEXO'] = sex_id
                                
                    df_asmr_temp['CONTEO_TOTAL'] = group['DEFUNCIONES'].sum()
                    asmr_estatal_list.append(df_asmr_temp)
                    
            if asmr_estatal_list:
                df_asmr_estatal = pd.concat(asmr_estatal_list, ignore_index=True)
                        
                tasas_asmr_estatal = pd.DataFrame({
                    'CIE10': cie10,
                    'ANIO': df_asmr_estatal['ANIO_REGIS'],
                    'ENFOQUE': 2,
                    'CVE_ENT': df_asmr_estatal['ENT_CVE'],
                    'CVE_MUN': None,
                    'SEXO': df_asmr_estatal['SEXO'],
                    'EDAD_INICIO': current_edad_ranges['EDAD_INICIO'].min(), 
                    'EDAD_FINAL': current_edad_ranges['EDAD_FINAL'].max(),    
                    'CONTEO': df_asmr_estatal['CONTEO_TOTAL'],
                    'TASA_CRUDA': 0,
                    'ASMR': df_asmr_estatal[asmr_column],
                    'ESCALA': estatal_config['code']
                })
                        
                all_tasas_asmr_estatal.append(tasas_asmr_estatal)
                print(f"      ASMR estatal procesado: {len(tasas_asmr_estatal)} registros")
        
        # ================================================================
        # TASAS ESTANDARIZADAS ASMR - NIVEL NACIONAL
        # ================================================================
        if len(df_nacional_crudo) > 0:
            print(f"    Calculando tasas estandarizadas - Nivel Nacional")
            std_name = 'WHO'        
            scale = '100K'          
            asmr_column = f"ASR({std_name})_{nacional_config['scale']}"

            who_nacional = ou.MortalityStandardizer(
                file_path=poblaciones_who,
                std_name=std_name,
                age_groups=current_age_groups.tolist()
            )

            asmr_nacional_list = []
            group_columns = ['ANIO_REGIS', 'SEXO'] if sex_id == 3 else ['ANIO_REGIS']
            
            for group_key, group in df_nacional_crudo.groupby(group_columns):
                if len(group) > 0:
                    df_asmr_temp = who_nacional.compute_ASR(
                        df=group[['ANIO_REGIS', 'RANGO_EDAD', 'TASA_CRUDA_100K']],
                        age_column="RANGO_EDAD",
                        rate_column="TASA_CRUDA_100K",
                        scale=scale
                    )
                    
                    if sex_id == 3:
                        anio, sexo = group_key
                        df_asmr_temp['ANIO_REGIS'] = anio
                        df_asmr_temp['SEXO'] = sexo
                    else:
                        anio = group_key if isinstance(group_key, (int, str)) else group_key[0]
                        df_asmr_temp['ANIO_REGIS'] = anio
                        df_asmr_temp['SEXO'] = sex_id
                        
                    df_asmr_temp['CONTEO_TOTAL'] = group['DEFUNCIONES'].sum()
                    asmr_nacional_list.append(df_asmr_temp)

            if asmr_nacional_list:
                df_asmr_nacional = pd.concat(asmr_nacional_list, ignore_index=True)
                
                tasas_asmr_nacional = pd.DataFrame({
                    'CIE10': cie10,
                    'ANIO': df_asmr_nacional['ANIO_REGIS'],
                    'ENFOQUE': 3,
                    'CVE_ENT': None,
                    'CVE_MUN': None,
                    'SEXO': df_asmr_nacional['SEXO'],
                    'EDAD_INICIO': current_edad_ranges['EDAD_INICIO'].min(),  
                    'EDAD_FINAL': current_edad_ranges['EDAD_FINAL'].max(),    
                    'CONTEO': df_asmr_nacional['CONTEO_TOTAL'],
                    'TASA_CRUDA': 0,  
                    'ASMR': df_asmr_nacional[asmr_column],
                    'ESCALA': nacional_config['code']
                })
                
                all_tasas_asmr_nacional.append(tasas_asmr_nacional)
                print(f"      ASMR nacional procesado: {len(tasas_asmr_nacional)} registros")
        

print("=== Ya acabó el bucle ===")

Procesando combinación 1/18 para longitud 1: grupos ['00_04']
  Rango de edades: 0-4
    Registros de muertes encontrados: 3576
    Calculando tasas crudas específicas - Nivel Municipal
      Datos municipales procesados: 2075 registros
    Calculando tasas crudas específicas - Nivel Estatal
      Datos estatales procesados: 1120 registros
    Calculando tasas crudas específicas - Nivel Nacional
      Datos nacionales procesados: 48 registros
    Calculando tasas estandarizadas - Nivel Municipal
      ASMR municipal procesado: 2075 registros
    Calculando tasas estandarizadas - Nivel Estatal


KeyboardInterrupt: 

In [ ]:
# ================================================================
# LIMPIAR VARIABLES ANTERIORES ANTES DE EJECUTAR EL BUCLE COMPLETO
# ================================================================

print("=== LIMPIANDO VARIABLES ===")

all_tasas_crudas_municipal = []
all_tasas_crudas_estatal = []
all_tasas_crudas_nacional = []
all_tasas_asmr_municipal = []
all_tasas_asmr_estatal = []
all_tasas_asmr_nacional = []

print("Variables limpiadas, listo para ejecutar el bucle completo integrado.")

=== LIMPIANDO VARIABLES DE EJECUCIONES ANTERIORES ===
Variables limpiadas, listo para ejecutar el bucle completo integrado.


In [ ]:
# ================================================================
# CONSOLIDACIÓN Y EXPORTACIÓN DE RESULTADOS EN UN SOLO ARCHIVO
# ================================================================

print("=== CONSOLIDANDO TODOS LOS RESULTADOS EN UN SOLO ARCHIVO ===")

# Output
output_folder = "../output"
os.makedirs(output_folder, exist_ok=True)

# Columnas clave para hacer el merge
merge_columns = ['CIE10', 'ANIO', 'ENFOQUE', 'CVE_ENT', 'CVE_MUN', 'SEXO', 'EDAD_INICIO', 'EDAD_FINAL', 'ESCALA']

# Lista para almacenar todos los DataFrames consolidados por nivel
all_consolidated_results = []

# ================================================================
# CONSOLIDAR NIVEL MUNICIPAL
# ================================================================
print("Consolidando nivel municipal...")

#  tasas crudas municipales
if all_tasas_crudas_municipal:
    df_crudas_municipal = pd.concat(all_tasas_crudas_municipal, ignore_index=True)
    df_crudas_municipal['TASA_CRUDA'] = df_crudas_municipal['TASA_CRUDA'].round(4)
    print(f"  Tasas crudas municipales: {len(df_crudas_municipal)} registros")
else:
    df_crudas_municipal = pd.DataFrame()

#  tasas ASMR municipales
if all_tasas_asmr_municipal:
    df_asmr_municipal = pd.concat(all_tasas_asmr_municipal, ignore_index=True)
    df_asmr_municipal['ASMR'] = df_asmr_municipal['ASMR'].round(4)
    print(f"  Tasas ASMR municipales: {len(df_asmr_municipal)} registros")
else:
    df_asmr_municipal = pd.DataFrame()

# merge a nivel municipal
if not df_crudas_municipal.empty and not df_asmr_municipal.empty:
    df_municipal_consolidado = pd.merge(
        df_crudas_municipal[merge_columns + ['CONTEO', 'TASA_CRUDA']], 
        df_asmr_municipal[merge_columns + ['CONTEO', 'ASMR']], 
        on=merge_columns, 
        how='outer',
        suffixes=('_cruda', '_asmr')
    )
    df_municipal_consolidado['CONTEO'] = df_municipal_consolidado['CONTEO_cruda'].fillna(df_municipal_consolidado['CONTEO_asmr'])
    df_municipal_consolidado = df_municipal_consolidado.drop(['CONTEO_cruda', 'CONTEO_asmr'], axis=1)
    df_municipal_consolidado['TASA_CRUDA'] = df_municipal_consolidado['TASA_CRUDA'].fillna(0)
    df_municipal_consolidado['ASMR'] = df_municipal_consolidado['ASMR'].fillna(0)
    all_consolidated_results.append(df_municipal_consolidado)
    print(f"  Municipal consolidado: {len(df_municipal_consolidado)} registros")
elif not df_crudas_municipal.empty:
    df_crudas_municipal['ASMR'] = 0
    all_consolidated_results.append(df_crudas_municipal)
    print(f"  Municipal (solo crudas): {len(df_crudas_municipal)} registros")
elif not df_asmr_municipal.empty:
    df_asmr_municipal['TASA_CRUDA'] = 0
    all_consolidated_results.append(df_asmr_municipal)
    print(f"  Municipal (solo ASMR): {len(df_asmr_municipal)} registros")

# ================================================================
# CONSOLIDAR NIVEL ESTATAL
# ================================================================
print("Consolidando nivel estatal...")

#  tasas crudas estatales
if all_tasas_crudas_estatal:
    df_crudas_estatal = pd.concat(all_tasas_crudas_estatal, ignore_index=True)
    df_crudas_estatal['TASA_CRUDA'] = df_crudas_estatal['TASA_CRUDA'].round(4)
    print(f"  Tasas crudas estatales: {len(df_crudas_estatal)} registros")
else:
    df_crudas_estatal = pd.DataFrame()

#  tasas ASMR estatales
if all_tasas_asmr_estatal:
    df_asmr_estatal = pd.concat(all_tasas_asmr_estatal, ignore_index=True)
    df_asmr_estatal['ASMR'] = df_asmr_estatal['ASMR'].round(4)
    print(f"  Tasas ASMR estatales: {len(df_asmr_estatal)} registros")
else:
    df_asmr_estatal = pd.DataFrame()

#  merge a nivel estatal
if not df_crudas_estatal.empty and not df_asmr_estatal.empty:
    df_estatal_consolidado = pd.merge(
        df_crudas_estatal[merge_columns + ['CONTEO', 'TASA_CRUDA']], 
        df_asmr_estatal[merge_columns + ['CONTEO', 'ASMR']], 
        on=merge_columns, 
        how='outer',
        suffixes=('_cruda', '_asmr')
    )
    df_estatal_consolidado['CONTEO'] = df_estatal_consolidado['CONTEO_cruda'].fillna(df_estatal_consolidado['CONTEO_asmr'])
    df_estatal_consolidado = df_estatal_consolidado.drop(['CONTEO_cruda', 'CONTEO_asmr'], axis=1)
    df_estatal_consolidado['TASA_CRUDA'] = df_estatal_consolidado['TASA_CRUDA'].fillna(0)
    df_estatal_consolidado['ASMR'] = df_estatal_consolidado['ASMR'].fillna(0)
    all_consolidated_results.append(df_estatal_consolidado)
    print(f"  Estatal consolidado: {len(df_estatal_consolidado)} registros")
elif not df_crudas_estatal.empty:
    df_crudas_estatal['ASMR'] = 0
    all_consolidated_results.append(df_crudas_estatal)
    print(f"  Estatal (solo crudas): {len(df_crudas_estatal)} registros")
elif not df_asmr_estatal.empty:
    df_asmr_estatal['TASA_CRUDA'] = 0
    all_consolidated_results.append(df_asmr_estatal)
    print(f"  Estatal (solo ASMR): {len(df_asmr_estatal)} registros")

# ================================================================
# CONSOLIDAR NIVEL NACIONAL
# ================================================================
print("Consolidando nivel nacional...")

#  tasas crudas nacionales
if all_tasas_crudas_nacional:
    df_crudas_nacional = pd.concat(all_tasas_crudas_nacional, ignore_index=True)
    df_crudas_nacional['TASA_CRUDA'] = df_crudas_nacional['TASA_CRUDA'].round(4)
    print(f"  Tasas crudas nacionales: {len(df_crudas_nacional)} registros")
else:
    df_crudas_nacional = pd.DataFrame()

#  tasas ASMR nacionales
if all_tasas_asmr_nacional:
    df_asmr_nacional = pd.concat(all_tasas_asmr_nacional, ignore_index=True)
    df_asmr_nacional['ASMR'] = df_asmr_nacional['ASMR'].round(4)
    print(f"  Tasas ASMR nacionales: {len(df_asmr_nacional)} registros")
else:
    df_asmr_nacional = pd.DataFrame()

#  merge a nivel nacional
if not df_crudas_nacional.empty and not df_asmr_nacional.empty:
    df_nacional_consolidado = pd.merge(
        df_crudas_nacional[merge_columns + ['CONTEO', 'TASA_CRUDA']], 
        df_asmr_nacional[merge_columns + ['CONTEO', 'ASMR']], 
        on=merge_columns, 
        how='outer',
        suffixes=('_cruda', '_asmr')
    )
    df_nacional_consolidado['CONTEO'] = df_nacional_consolidado['CONTEO_cruda'].fillna(df_nacional_consolidado['CONTEO_asmr'])
    df_nacional_consolidado = df_nacional_consolidado.drop(['CONTEO_cruda', 'CONTEO_asmr'], axis=1)
    df_nacional_consolidado['TASA_CRUDA'] = df_nacional_consolidado['TASA_CRUDA'].fillna(0)
    df_nacional_consolidado['ASMR'] = df_nacional_consolidado['ASMR'].fillna(0)
    all_consolidated_results.append(df_nacional_consolidado)
    print(f"  Nacional consolidado: {len(df_nacional_consolidado)} registros")
elif not df_crudas_nacional.empty:
    df_crudas_nacional['ASMR'] = 0
    all_consolidated_results.append(df_crudas_nacional)
    print(f"  Nacional (solo crudas): {len(df_crudas_nacional)} registros")
elif not df_asmr_nacional.empty:
    df_asmr_nacional['TASA_CRUDA'] = 0
    all_consolidated_results.append(df_asmr_nacional)
    print(f"  Nacional (solo ASMR): {len(df_asmr_nacional)} registros")

# ================================================================
# CONSOLIDAR TODO EN UN ARCHIVO FINAL
# ================================================================
if all_consolidated_results:
    df_consolidado = pd.concat(all_consolidated_results, ignore_index=True)
    
    # Ordenar por las columnas clave
    df_consolidado = df_consolidado.sort_values([
        'ENFOQUE', 'ANIO', 'CVE_ENT', 'CVE_MUN', 'SEXO', 'EDAD_INICIO'
    ], na_position='last').reset_index(drop=True)
    
    # Reordenar columnas en el orden deseado
    column_order = ['CIE10', 'ANIO', 'ENFOQUE', 'CVE_ENT', 'CVE_MUN', 'SEXO', 
                    'EDAD_INICIO', 'EDAD_FINAL', 'CONTEO', 'TASA_CRUDA', 'ASMR', 'ESCALA']
    df_consolidado = df_consolidado[column_order]
    
    print(f"\n=== RESUMEN DEL ARCHIVO CONSOLIDADO ===")
    print(f"Total de registros: {len(df_consolidado):,}")
    print(f"Distribución por ENFOQUE:")
    print(df_consolidado['ENFOQUE'].value_counts().sort_index())
    
    # Mostrar estadísticas de completitud
    print(f"\nEstadísticas de completitud:")
    print(f"Registros con TASA_CRUDA > 0: {(df_consolidado['TASA_CRUDA'] > 0).sum():,}")
    print(f"Registros con ASMR > 0: {(df_consolidado['ASMR'] > 0).sum():,}")
    print(f"Registros con ambas tasas > 0: {((df_consolidado['TASA_CRUDA'] > 0) & (df_consolidado['ASMR'] > 0)).sum():,}")
    
    print(f"\nPrimeras 10 filas del archivo:")
    display(df_consolidado.head(10))
    
    # Exportar archivo consolidado
    output_file = os.path.join(output_folder, "tasas_mortalidad_consolidado.csv")
    df_consolidado.to_csv(output_file, index=False)
    
    file_size = os.path.getsize(output_file)
    print(f"\n=== ARCHIVO EXPORTADO ===")
    print(f"Archivo: {output_file}")
    print(f"Tamaño: {file_size:,} bytes")
    print(f"Registros: {len(df_consolidado):,}")
    
else:
    print("No se generaron datos")

print("\n=== Ya acab[e] ===")